# Quality control on a Bosch assembly line — Six Sigma / DMAIC on big data

**Author:** Esteban López · Industrial Engineer  
**Dataset:** Bosch Production Line Performance — Kaggle (14.3 GB). 1,184,687 real parts, 968 numeric measurements across 51 stations on 4 production lines (L0–L3), plus date and categorical features. Binary `Response` (1 = fails quality control).  
**Source:** https://www.kaggle.com/c/bosch-production-line-performance

## Project goal (Six Sigma / continuous improvement)
Use the **DMAIC** cycle on real, large-scale manufacturing data to: (1) map the value stream from the station naming + date features, (2) measure the process baseline and its extreme imbalance, (3) find where defects concentrate, and (4) build a defect-prediction model evaluated with **MCC** — the right metric when only 0.58% of parts fail.

> **Reproducible** notebook. Data via the Kaggle API (`kaggle competitions download -c bosch-production-line-performance`). The dataset is huge, so the code reads in **chunks** and **down-casts** dtypes — a deliberate big-data engineering choice.

**Key engineering point:** with a 0.58% defect rate, *accuracy is meaningless* (predicting "all pass" scores 99.4%). The competition metric — and the right one for rare-event quality — is the **Matthews Correlation Coefficient (MCC)**.

## 0. Setup
Requirements: `pip install pandas numpy scikit-learn matplotlib`. Place `train_numeric.csv` (and optionally `train_date.csv`) in the working folder.

In [ ]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (9, 5), 'font.size': 11})
NAVY, CYAN, GOLD = '#0b1f3a', '#16b3c7', '#f2c14e'
NUMERIC = 'train_numeric.csv'   # 1,184,687 rows x (Id + 968 features + Response)

## 1. DEFINE / MEASURE — Read 14 GB without melting the laptop
We never load the full numeric matrix into memory. We scan the header to type every feature as `float32` and stream the file in chunks.

In [ ]:
# Build a dtype map: Id->int32, Response->int8, every L#_S#_F# -> float32 (halves the RAM vs float64)
header = pd.read_csv(NUMERIC, nrows=0).columns.tolist()
feat_cols = [c for c in header if c not in ('Id', 'Response')]
dtypes = {c: 'float32' for c in feat_cols}
dtypes['Id'] = 'int32'; dtypes['Response'] = 'int8'
print(f'Columns: {len(header)}  |  numeric features: {len(feat_cols)}')

In [ ]:
# Stream the file: count rows, failures, and per-station 'visits' (a part visits a station if it has
# any non-null measurement there). This is the basis of the value-stream map.
def station_of(col):
    m = re.match(r'(L\d+)_(S\d+)_', col)
    return (m.group(1), m.group(2)) if m else (None, None)

line_of   = {c: station_of(c)[0] for c in feat_cols}
stat_of   = {c: station_of(c)[1] for c in feat_cols}
stations  = sorted(set((line_of[c], stat_of[c]) for c in feat_cols))

n_rows = 0; n_fail = 0
visit_count = {s: 0 for s in stations}      # parts that pass through each station
visit_fail  = {s: 0 for s in stations}      # of those, how many fail

for chunk in pd.read_csv(NUMERIC, dtype=dtypes, chunksize=100_000):
    n_rows += len(chunk); n_fail += int(chunk['Response'].sum())
    resp = chunk['Response'].values.astype(bool)
    for (ln, st) in stations:
        cols = [c for c in feat_cols if line_of[c] == ln and stat_of[c] == st]
        visited = chunk[cols].notna().any(axis=1).values
        visit_count[(ln, st)] += int(visited.sum())
        visit_fail[(ln, st)]  += int((visited & resp).sum())

print(f'Parts: {n_rows:,}  |  Failures: {n_fail:,}  ({n_fail/n_rows*100:.2f}%)')
print(f'Defect rate: 1 in {n_rows//n_fail} parts')

## 2. ANALYZE — Value-stream map: where the process is measured and where it fails

In [ ]:
# Features per line (measurement density) -> a Pareto of where the line is most instrumented
per_line = pd.Series({c: line_of[c] for c in feat_cols}).value_counts().sort_values(ascending=False)
cum = per_line.cumsum() / per_line.sum() * 100
fig, ax1 = plt.subplots(figsize=(8.5, 5))
prev = [0] + list(cum.values[:-1])
ax1.bar(per_line.index, per_line.values, color=[CYAN if p < 80 else '#9fb3c8' for p in prev], edgecolor='white')
for i, v in enumerate(per_line.values):
    ax1.text(i, v + 5, str(v), ha='center', fontweight='bold', color=NAVY)
ax1.set_ylabel('Numeric measurements'); ax1.set_xlabel('Production line')
ax1.set_title('Where the process is measured (968 sensors / 4 lines)', fontweight='bold', color=NAVY)
ax2 = ax1.twinx(); ax2.plot(range(len(per_line)), cum.values, color=GOLD, marker='o', lw=2.4)
ax2.axhline(80, color='grey', ls='--'); ax2.set_ylabel('Cumulative %'); ax2.set_ylim(0, 108)
plt.tight_layout(); plt.show()
print('Vital-few lines:', list(cum[cum <= 80.5].index), '+ the line that crosses 80%')

In [ ]:
# Station-level defect rate (the heart of the VSM): where do failing parts concentrate?
vsm = pd.DataFrame({
    'line':   [s[0] for s in stations],
    'station':[s[1] for s in stations],
    'parts':  [visit_count[s] for s in stations],
    'fails':  [visit_fail[s]  for s in stations],
})
vsm['defect_rate_%'] = (vsm['fails'] / vsm['parts'].replace(0, np.nan) * 100).round(3)
vsm = vsm[vsm['parts'] > 1000].sort_values('defect_rate_%', ascending=False)
print('Top 10 stations by defect rate (among parts that pass through them):')
vsm.head(10)

**Lean reading:** the station naming `L{line}_S{station}_F{feature}` *is* a value-stream map. Lines L1 and L3 hold ~78% of all measurements (the most complex, most-monitored stages), and the station-level defect rate tells us which stations to put under SPC first.

## 3. IMPROVE — Defect-prediction model (evaluated with MCC)
Two engineering choices for this rare-event, huge, sparse problem:
- **HistGradientBoosting** handles NaN natively (each part only visits some stations, so most features are missing) and scales to millions of rows.
- We **down-sample the majority** class for training speed and judge on **MCC**, not accuracy.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef, confusion_matrix, classification_report

# Build a training sample: ALL failures + a random sample of passes (keeps signal, cuts size)
rng = np.random.RandomState(42)
fails_df, pass_keep = [], []
for chunk in pd.read_csv(NUMERIC, dtype=dtypes, chunksize=200_000):
    fails_df.append(chunk[chunk['Response'] == 1])
    passes = chunk[chunk['Response'] == 0]
    pass_keep.append(passes.sample(frac=0.05, random_state=42))   # ~5% of passes
sample = pd.concat(fails_df + pass_keep, ignore_index=True)
X = sample[feat_cols]; y = sample['Response']
print('Training sample:', X.shape, '| positives:', int(y.sum()))

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1,
                                     l2_regularization=1.0, class_weight='balanced', random_state=42)
clf.fit(Xtr, ytr)

In [ ]:
# Tune the decision threshold to MAXIMISE MCC (the lever that matters here)
proba = clf.predict_proba(Xte)[:, 1]
ths = np.linspace(0.05, 0.95, 37)
mccs = [matthews_corrcoef(yte, (proba >= t).astype(int)) for t in ths]
best_t = ths[int(np.argmax(mccs))]
pred = (proba >= best_t).astype(int)
print(f'Best threshold: {best_t:.2f}  |  MCC: {max(mccs):.3f}')
print('\n', classification_report(yte, pred, target_names=['Pass', 'Fail']))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ths, mccs, color=CYAN, lw=2); ax.axvline(best_t, color=GOLD, ls='--')
ax.set_xlabel('Decision threshold'); ax.set_ylabel('MCC'); ax.set_title('Threshold tuning for MCC', fontweight='bold', color=NAVY)
plt.tight_layout(); plt.show()

In [ ]:
cm = confusion_matrix(yte, pred)
fig, ax = plt.subplots(figsize=(5, 4.4)); ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_xticklabels(['Pass','Fail']); ax.set_yticks([0,1]); ax.set_yticklabels(['Pass','Fail'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=12, color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('Confusion matrix', fontweight='bold', color=NAVY)
plt.tight_layout(); plt.show()

## 4. CONTROL — From model to line control

1. **Focus SPC where the process is complex.** Lines L1 and L3 carry ~78% of measurements — put the highest-defect-rate stations there under control charts first.
2. **Catch escapes, accept some false alarms.** Tune the threshold to the MCC optimum (or to the cost ratio of a missed defect vs. an extra check) — never to accuracy.
3. **Trace every failing part's path.** The date features give each part's route; recurring routes among failures point to a specific station/sequence to fix.
4. **Poka-yoke at the critical stations.** Convert the model's top signals into in-line gates so defects are stopped, not just predicted.
5. **PDCA at scale.** Re-score MCC and the station defect map per shift; re-train as the line drifts.

*Note on metric:* a single HistGradientBoosting model typically lands in the MCC ~0.40–0.48 range on this dataset; the Kaggle-winning ensembles reached ~0.52. The point of this case study is the **engineering workflow** — big-data handling, value-stream thinking and the right metric — not a leaderboard score.